# Access to Alerts at USDF,  view images and Reprocess to calculate the fluxes using standard DRP pipeline

- **Author:** Sylvie Dagoret-Campagne  
- **Creation date:** 2026-04-15
- **Last update:** 2026-04-16
- **Environment:** USDF RSP — `LSST` kernel (`lsst_distrib` stack)
- **From DP1 tutorial:** [105.2. Detect and measure sources](https://dp1.lsst.io/tutorials/notebook/105/notebook-105-2.html)

- [yaml configuration file for prompt proccesing](https://tigress-web.princeton.edu/~lkelvin/pipelines/current/ap_pipe/LSSTCam/ApPipe/pipeline_ap_pipe_LSSTCam_ApPipe.yaml)
- [processing diagrams](https://tigress-web.princeton.edu/~lkelvin/pipelines/current/ap_pipe/LSSTCam/ApPipe/pipeline_ap_pipe_LSSTCam_ApPipe.pdf)
- [code on prompt processing](https://github.com/lsst/pipe_tasks/blob/253578fa50f35f49b5e34252c1b73620e36ab593/python/lsst/pipe/tasks/matchDiffimSourceInjected.py#L143) 

```yaml
prompt:
    subset:
    - filterDiaSource
    - analyzeImageDifferenceMetrics
    - detectAndMeasureDiaSource
    - analyzeDiaSourceAssociationMetrics
    - analyzePreliminarySummaryStats
    - isr
    - subtractImages
    - standardizeDiaSource
    - analyzeDiaSourceDetectionMetrics
    - associateApdb
    - filterDiaSourcePostReliability
    - calibrateImage
    - analyzeAssociateDiaSourceTiming
    - buildTemplate
    - analyzeAssociatedDiaSourceTable
    - computeReliability
    - analyzeTrailedDiaSourceTable
    description: 'Tasks necessary to turn raw images into APDB rows and alerts. Requires
      preload subset to be run first.
```

## 1. Imports

Standard scientific stack plus the LSST middleware:  
- `lsst.daf.butler` — data access layer  
- `lsst.afw.display` — image display utilities  
- `lsst.geom` — sky coordinate geometry  
- `lsst.skymap` — tract/patch sky tessellation

In [ ]:
import lsst.pipe.base

print(lsst.pipe.base.__version__)

In [ ]:
import sys
import matplotlib.pyplot as plt

import lsst.pipe.base
import lsst.daf.base as dafBase
from lsst.daf.butler import Butler

import lsst.afw.display as afwDisplay
import lsst.afw.table as afwTable
from lsst.afw.image import ExposureF

import lsst.geom as geom
from lsst.geom import SpherePoint, degrees

from lsst.skymap import PatchInfo, Index2D
import numpy as np
import pandas as pd
from astropy.time import Time

from lsst.meas.algorithms.detection import SourceDetectionTask
from lsst.meas.base import SingleFrameMeasurementTask
from lsst.afw.table import SourceTable
# from lsst.meas.deblender import SourceDeblendTask


# %matplotlib widget
import seaborn as sns
# from lsst.analysis.ap import apdb

# import lsst.afw.display as afwDisplay
# from firefly_client import FireflyClient

# import lsst.afw.image as afwImage
import math

In [ ]:
def get_uris_from_butler(butler, datasetType, collections=None, **query_kwargs):
    """
    Return a list of URIs for a given datasetType in a Butler collection.

    Parameters
    ----------
    butler : lsst.daf.butler.Butler
        The Butler object
    datasetType : str
        The dataset type, e.g. 'raw', 'calexp'
    collections : list of str, optional
        Butler collections to query
    query_kwargs : dict
        Additional query parameters, e.g. visit=123, filter='r'

    Returns
    -------
    uris : list of str
        File paths of the datasets
    """
    refs = butler.registry.queryDatasets(datasetType, collections=collections, **query_kwargs)
    return [butler.getURI(ref) for ref in refs]

In [ ]:
def get_refs_from_butler(butler, datasetType, collections=None, return_uris=False, **query_kwargs):
    """
    Get dataset references (or URIs) from a Butler, safely handling numpy types.

    Parameters
    ----------
    butler : lsst.daf.butler.Butler
        Butler instance.
    datasetType : str
        Dataset type to query (e.g., "calexp", "raw", "src").
    collections : str or list of str, optional
        Butler collections to query.
    return_uris : bool, default False
        If True, return URIs instead of refs.
    **query_kwargs :
        Query parameters, e.g., visit=[...], ccd=[...], filter=['r','g'].

    Returns
    -------
    list of DatasetRef or list of str
        List of references or URIs.
    """

    # Convert numpy types to Python native types to avoid Butler query errors
    safe_kwargs = {}
    for k, v in query_kwargs.items():
        if isinstance(v, np.ndarray):
            safe_kwargs[k] = [int(x) if np.issubdtype(type(x), np.integer) else x for x in v]
        elif isinstance(v, (np.integer, np.int64, np.int32)):
            safe_kwargs[k] = int(v)
        else:
            safe_kwargs[k] = v

    # Query dataset references
    refs = butler.registry.queryDatasets(datasetType, collections=collections, **safe_kwargs)

    if return_uris:
        return [butler.getURI(ref) for ref in refs]

    return refs

## 2. Configuration

Select the Butler repository, the DRP processing collection, the sky map name,  
and the instrument.  Multiple DRP collections are listed for reference:  
uncomment the desired one before running.


- See collection here : https://usdf-rsp.slac.stanford.edu/plot-navigator
- See Campaign : https://rubinobs.atlassian.net/wiki/spaces/DM/pages/661192727/LSSTCam+Intermittent+DRP+Runs
- See the bulter lists : https://developer.lsst.io/usdf/storage.html
- See Prompt+Processing+with+LSSTCam : https://rubinobs.atlassian.net/wiki/spaces/DM/pages/1417609236/Prompt+Processing+with+LSSTCam+Daytime+AP+2026#
- Examples of Notebooks : https://github.com/lsst-so/notebooks_dia/tree/main

In [ ]:
# ── Butler repository and DRP collection ──────────────────────────────────────
REPO = "main"
collection = "LSSTCam/prompt/output-2026-02-26"
skymapName = "lsst_cells_v2"
instrument = "LSSTCam"

# ── Date range for exposure queries ───────────────────────────────────────────
date_start = 20260101  # YYYYMMDD — earliest day_obs to include
date_selection = 20260409  # YYYYMMDD — reference date for the analysis

# ── Build WHERE clauses for Butler registry queries ───────────────────────────
where_clause = "instrument = '" + f"{instrument}" + "'"
where_clause_date = where_clause + f"and day_obs >= {date_start}"

# ── Instantiate the Butler and its registry ───────────────────────────────────
butler = Butler(REPO, collections=collection)
registry = butler.registry

In [ ]:
# Retrieve the sky tessellation (skyMap) from the butler.
# The 'lsst_cells_v2' skymap divides the full sky into tracts and patches.
# Error handling is included because the skymap may not be present in all collections.
try:
    skymap = butler.get("skyMap", skymap=skymapName, collections=collection)
except Exception as inst:
    print(type(inst))  # exception type
    print(inst.args)  # arguments stored in .args
    print(inst)  # full message via __str__

In [ ]:
# Inspect the skymap object (number of tracts, pixel scale, etc.)
# skymap

In [ ]:
FLAGS_DEBUG_SCHEMA = False

## 8. Available dataset types in the collection

List all dataset types present in the selected collection,  
filtering out pipeline-internal products (configs, logs, metrics, plots)  
to focus on science data products (calexp, src, dia_source, dia_object, …).


In [ ]:
FLAG_DUMP_DATASETS = True
if FLAG_DUMP_DATASETS:
    for datasetType in registry.queryDatasetTypes():
        if registry.queryDatasets(datasetType, collections=collection).any(execute=False, exact=False):
            # Skip pipeline bookkeeping products
            if (
                ("_config" not in datasetType.name)
                and ("_log" not in datasetType.name)
                and ("_metadata" not in datasetType.name)
                and ("_resource_usage" not in datasetType.name)
                and ("Plot" not in datasetType.name)
                and ("Metric" not in datasetType.name)
                and ("metric" not in datasetType.name)
            ):
                print(datasetType)

In [ ]:
# Inspect the required dimensions for dia_source (needed for difference imaging queries)
print(butler.registry.getDatasetType("dia_forced_source_apdb").dimensions)

In [ ]:
# Inspect the required dimensions for dia_object
print(butler.registry.getDatasetType("dia_object_apdb").dimensions)

In [ ]:
# Inspect the required dimensions for dia_source (needed for difference imaging queries)
print(butler.registry.getDatasetType("dia_source_apdb").dimensions)

In [ ]:
# Inspect the required dimensions for dia_source (needed for difference imaging queries)
print(butler.registry.getDatasetType("new_dia_source").dimensions)

In [ ]:
refs = list(
    butler.registry.queryDatasets(
        "prompt_provenance",
        collections=collection,
    )
)
all_dataId = []
for ref in refs[:10]:
    print(ref.dataId)
    all_dataId.append(ref.dataId)

In [ ]:
the_dataId = all_dataId[0]
print(the_dataId)

In [ ]:
# import lsst.pipe.base.graph
# import lsst.pipe.base.graph.graph
# from lsst.daf.butler import StorageClassFactory
# import lsst.daf.butler
# import lsst.pipe.base
# import lsst.ctrl.mpexec
# import lsst.pipe.base.cli.cmd

# scf = StorageClassFactory()
# print("ProvenanceQuantumGraph" in scf)

In [ ]:
# dt = butler.get_dataset_type("prompt_provenance")
# print(dt.storageClass)

In [ ]:
# print(f"the_dataId = {the_dataId}")
# prov = butler.get("prompt_provenance", dataId=the_dataId)
# qgraph = prov.quantumGraph  # ou équivalent selon version

# for quantum in qgraph:
#    print(quantum.taskName)
#    print(quantum.dataId)

## Example of Light curve subsample in Fink-Broker data

```
 	diaObj 	diaSrc 	mjd 	band 	visit 	detector 	x 	y 	psfFlux 	psfFluxErr 	scienceFlux 	scienceFluxErr 	templateFlux 	templateFluxErr 	day_obs 	month_obs
1029 	313853517444939794 	170063690301177938 	61098.108349 	y 	2026022600207 	51 	145.16164 	3639.300500 	10921.9270 	1243.32760 	44045.580 	1226.78440 	32617.346 	274.21304 	20260226 	202602
1030 	313853517444939794 	170063690452172875 	61098.108813 	y 	2026022600208 	83 	3827.32930 	2687.968000 	9844.3880 	1189.71690 	42716.867 	1176.07300 	32472.490 	273.70413 	20260226 	202602
1031 	313853517444939794 	170063690858496056 	61098.112664 	y 	2026022600211 	90 	2788.16190 	1668.272300 	10380.5080 	1019.47400 	43284.140 	1013.96420 	31991.410 	376.74704 	20260226 	202602
1032 	313853517444939794 	170063690994286595 	61098.113128 	y 	2026022600212 	93 	1259.44450 	41.697483 	10514.9760 	1008.98395 	42434.120 	998.61346 	30744.092 	393.22940 	20260226 	202602
1033 	313853517444939794 	170063723341807659 	61098.270223 	y 	2026022600453 	95 	3550.78900 	1363.893300 	8229.4970 	1017.97644 	40883.297 	1023.03370 	31669.326 	411.03214 	20260226 	202602
1034 	313853517444939794 	170063723474452578 	61098.270691 	y 	2026022600454 	92 	2100.58940 	2938.168500 	9661.8140 	1010.44160 	41524.082 	1016.33514 	30815.621 	452.85654 	20260226 	202602
1035 	313853517444939794 	170063723613913266 	61098.272697 	r 	2026022600455 	102 	2612.80860 	3824.363000 	8016.3496 	189.21060 	25180.540 	187.87000 	17067.248 	39.82949 	20260226 	202602
1036 	313853517444939794 	170063723748130852 	61098.273166 	r 	2026022600456 	102 	1098.03020 	841.125730 	7928.1895 	188.90501 	25311.723 	189.69778 	17217.742 	64.94376 	20260226 	202602
1037 	313853517444939794 	170063723878678545 	61098.275115 	z 	2026022600457 	95 	1300.34720 	544.580300 	9032.4520 	417.41107 	30190.838 	413.37164 	21071.193 	122.68716 	20260226 	202602
1038 	313853517444939794 	170063724011323506 	61098.275586 	z 	2026022600458 	92 	3572.64770 	3112.698700 	9296.9560 	409.83118 	30139.334 	404.60516 	20683.617 	138.13329 	20260226 	202602

```

### Visit in Fink-Broker

In [ ]:
diaObjectId = 313853517444939794
dataId = {"instrument": instrument, "band": "r", "detector": 102, "visit": 2026022600455}

#### new sources

In [ ]:
t1 = butler.get("new_dia_source", dataId)

In [ ]:
t1

#### dia_source_detector

In [ ]:
t2 = butler.get("dia_source_detector", dataId)

- the psfFlux is the one expected in the butler
- are we abel to recompute the same flux at this position ???

In [ ]:
t2

#### all sources

In [ ]:
t3 = butler.get("dia_source_apdb", dataId)

#### study this source "0" en remember (x0,y0,ra0,dec0)

In [ ]:
src0 = t3[t3["diaObjectId"] == diaObjectId]
src0

In [ ]:
# extract sr0 variable subset
sr0 = src0[
    ["x", "y", "diaSourceId", "visit", "detector", "psfFlux", "apFlux", "scienceFlux", "snr", "ra", "dec"]
]
src0

In [ ]:
# remember postion
x0 = src0["x"]
y0 = x0 = src0["y"]
ra0 = src0["ra"]
dec0 = src0["dec"]

### get the footprints

In [ ]:
t_star_footprints = butler.get("single_visit_star_footprints", dataId)

In [ ]:
t_star_footprints

## View the Science, DIA and Templates images

In [ ]:
try:
    the_singlevisitimage = butler.get("preliminary_visit_image", dataId)

    the_backgroundimage = butler.get("preliminary_visit_image_background", dataId)
    the_differenceimage = butler.get("difference_image", dataId)
    the_differenceimagemask = the_differenceimage.getMaskedImage()
    the_template_detector = butler.get("template_detector", dataId)
    the_title = f"{dataId["visit"]},{dataId["detector"]},{dataId["band"]}"
    x, y = t3["x"][0], t3["y"][0]
    list_of_points = [(x, y)]
except Exception as e:
    # log utile mais compact
    print(f"[FAIL] → visit={dataId['visit']} det={dataId['detector']} band={dataId['band']}")
    print(f"       {type(e).__name__}: {e}")

In [ ]:
all_singlevisitimages = [the_singlevisitimage]
all_backgroundimages = [the_backgroundimage]
all_differenceimages = [the_differenceimage]
all_differenceimagesmask = [the_differenceimagemask]
all_template_detector = [the_template_detector]

all_titles = [the_title]
all_datapoints = [list_of_points]
N = len(all_singlevisitimages)
print(f"Number of image to plot : {N}")
count = 0

### Plot in firefly

In [ ]:
afwDisplay.setDefaultBackend("firefly")

In [ ]:
for count in range(N):
    # display the science image
    display = afwDisplay.Display(frame=count + 1)
    # cannot succeed to show white stars on dark sky
    # display.setImageColormap('gray')
    display.scale("asinh", "zscale")
    display.mtv(all_singlevisitimages[count].image, title=all_titles[count])
    list_of_points = all_datapoints[count]
    with display.Buffering():
        #
        for x, y in list_of_points:
            display.dot("o", x, y, size=30, ctype="red")

    # display the background image
    display = afwDisplay.Display(frame=count + 2)
    # cannot succeed to show white stars on dark sky
    # display.setImageColormap('gray')
    # display.scale("asinh", "zscale")
    display.mtv(all_backgroundimages[count].getImage(), title=all_titles[count] + "_back")

    # display the difference image
    display = afwDisplay.Display(frame=count + 3)
    display.scale("asinh", "zscale")
    display.mtv(all_differenceimages[count].image, title=all_titles[count] + "_dia")
    list_of_points = all_datapoints[count]
    with display.Buffering():
        #
        for x, y in list_of_points:
            display.dot("o", x, y, size=30, ctype="blue")

    # display the dia image mask
    display = afwDisplay.Display(frame=count + 4)
    # display.scale("asinh", "zscale")
    display.mtv(all_differenceimagesmask[count].image, title=all_titles[count] + "_dia_mask")

    # display the template image
    display = afwDisplay.Display(frame=count + 5)
    display.scale("asinh", "zscale")
    display.mtv(all_template_detector[count].image, title=all_titles[count] + "_temp")
    list_of_points = all_datapoints[count]
    with display.Buffering():
        #
        for x, y in list_of_points:
            display.dot("o", x, y, size=30, ctype="green")

## DIA image analysis

In [ ]:
# Not doable in a notebook
# from lsst.ap.association import DiaPipelineTask
# task = DiaPipelineTask()

### Photocalib

In [ ]:
photoCalib = the_differenceimage.getPhotoCalib()
photoCalib.getInstFluxAtZeroMagnitude()

In [ ]:
photoCalib.instFluxToMagnitude(3630780547701.0024)

## Goal Calculate myslef the fluxes for that visit from the DIA image 

==> **Prepare the image for re-processing**

In [ ]:
img = the_differenceimage.image.array

print(img.min(), img.max())

### Explore the concept of shema and table for sources

In [ ]:
# the_differenceimage.removeAndClearMaskPlane('DETECTED')

#### Define a minimal Source table schema as an example

In [ ]:
schema = afwTable.SourceTable.makeMinimalSchema()
print(schema)

#### Add missing fields to the schema

In [ ]:
raerr = schema.addField("coord_raErr", type="F")
decerr = schema.addField("coord_decErr", type="F")

In [ ]:
# print(schema.getAliasMap())

### Create a container which will be used to record metadata about algorithm execution.

In [ ]:
algMetadata = dafBase.PropertyList()

### Source Detection task configuration
- note in that example one uses the minimal sheme only as an example but we will not use this minimal scheme later

In [ ]:
config = SourceDetectionTask.ConfigClass()
config.thresholdValue = 5
config.thresholdType = "stdev"
config.thresholdPolarity = "both"
sourceDetectionTask = SourceDetectionTask(schema=schema, config=config)
del config

### SingleFrameMeasurementTask Configuration
- note in that example one uses the minimal sheme only as an example but we wil not use this minimal schem a later

In [ ]:
config = SingleFrameMeasurementTask.ConfigClass()
sourceMeasurementTask = SingleFrameMeasurementTask(schema=schema, config=config, algMetadata=algMetadata)
del config

## Do the proper configuration with the correct shema for the sources then Run the sources detection and measurements Tasks

```
✔ task → définit schema
✔ detection → remplit catalog conforme
✔ measurement → exige EXACTEMENT ce schema
```

#### The sourceMeasurementTask define the Source scheme it requires

- we need here a more complex sheme than the minimal scheme. It is the soureMeasurementTask that really define the sourceTbale schema it requires

In [ ]:
schema = sourceMeasurementTask.schema
tabl = SourceTable.make(schema)

In [ ]:
print(schema.getNames())

In [ ]:
flag_shema_is_negative = False
if "is_negative" in schema.getNames():
    flag_shema_is_negative = True
else:
    print('be carefull no "is_negative" ')

### redo the config for the detection task

In [ ]:
config = SourceDetectionTask.ConfigClass()
config.thresholdValue = 5
config.thresholdType = "stdev"
if not flag_shema_is_negative:
    config.thresholdPolarity = "both"
config.reEstimateBackground = False  # souvent important en DIA
sourceDetectionTask = SourceDetectionTask(schema=schema, config=config)
del config

In [ ]:
# print("TASK schema:", sourceMeasurementTask.schema)

#### Run the source detection

In [ ]:
resultsDet = sourceDetectionTask.run(tabl, the_differenceimage)

In [ ]:
# resultsDet.numPosPeaks
sources = resultsDet.sources
# print(len(sources))

### Redo the sourceMeasurement task config

In [ ]:
# config = SingleFrameMeasurementTask.ConfigClass()

In [ ]:
# print(config)

In [ ]:
# sourceMeasurementTask = SingleFrameMeasurementTask(schema=schema, config=config, algMetadata=algMetadata)
# sourceMeasurementTask = SingleFrameMeasurementTask(schema=schema, config=config)
# del config

#### Run the measurements (probably the fluxes)

In [ ]:
resultsMeas = sourceMeasurementTask.run(sources, the_differenceimage)

In [ ]:
# problem all sources are positives !
for src in sources:
    fp = src.getFootprint()
    peaks = fp.getPeaks()

    for p in peaks:
        print(p.getIx(), p.getIy(), p.getPeakValue())

### Check the fluxes

In [ ]:
for qqf in ["base_PsfFlux_instFlux", "slot_PsfFlux_instFlux", "slot_CalibFlux_instFlux"]:
    fluxes = sources[qqf]

    print(f"{qqf} : --> (min,max) ", np.min(fluxes), np.max(fluxes))

In [ ]:
# print("CAT schema:", sources.getSchema())
# print("TASK schema:", sourceMeasurementTask.schema)

In [ ]:
keywords = ["x", "y", "ra", "dec"]

for name in sources.schema.getNames():
    if any(k in name.lower() for k in keywords):
        print(name)

In [ ]:
sources.asAstropy()[:5]

### Get footprints

In [ ]:
for src in sources:
    fp = src.getFootprint()
    if fp is None:
        continue

    spans = fp.getSpans()

In [ ]:
spans

### Plot in firefly

In [ ]:
display = afwDisplay.Display(frame=count + 6)
display.scale("asinh", "zscale")
display.mtv(the_differenceimage.image, title="footprints dia")

with display.Buffering():
    for src in sources:
        fp = src.getFootprint()
        if fp is None:
            continue

        display.dot("o", src.getX(), src.getY(), size=5, ctype="red")
        display.dot("+", src.getX(), src.getY(), size=10, ctype="red")

### Dump source table

In [ ]:
sources.asAstropy()

### My dump

In [ ]:
photoCalib = the_differenceimage.getPhotoCalib()
for idx, src in enumerate(sources):
    base_PsfFlux_instFlux = src.get("base_PsfFlux_instFlux")
    base_PsfFlux_instFluxErr = src.get("base_PsfFlux_instFluxErr")
    base_PsfFlux_instMag = photoCalib.instFluxToMagnitude(base_PsfFlux_instFlux)
    print(
        f"{idx} PsfFlux_instFlux = {base_PsfFlux_instFlux:.2f} +/-  {base_PsfFlux_instFluxErr:.2f} nJy (m = {base_PsfFlux_instMag:.2f})"
    )

### Check the flux

In [ ]:
df = sources.asAstropy().to_pandas()
df.head()

In [ ]:
wcs = the_differenceimage.getWcs()

for src in sources:
    sky = wcs.pixelToSky(src.getCentroid())
#    src.set("coord_ra", sky.getRa().asDegrees())
#    src.set("coord_dec", sky.getDec().asDegrees())

In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u

center = SkyCoord(ra0, dec0, unit="deg")
coords = SkyCoord(df["coord_ra"], df["coord_dec"], unit="deg")
sep = coords.separation(center)
sep.min()

In [ ]:
dx = df["slot_Centroid_x"] - x0
dy = df["slot_Centroid_y"] - y0

df["dist"] = np.sqrt(dx**2 + dy**2)

In [ ]:
df["dist"].min()

### Plot the footprints

#### Compute scaling and normalisation of the image

In [ ]:
img = the_differenceimage.image.array
vmin, vmax = np.percentile(img, (2, 98))
stretch = 3.0  # contrôle dynamique
norm_img = (img - vmin) / (vmax - vmin)
norm_img = np.clip(norm_img, 0, 1)
asinh_img = np.arcsinh(stretch * norm_img) / np.arcsinh(stretch)

In [ ]:
from astropy.visualization import AsinhStretch, ImageNormalize, ZScaleInterval

norm_old = ImageNormalize(img, interval=ZScaleInterval(), stretch=AsinhStretch())

In [ ]:
norm = ImageNormalize(img, interval=ZScaleInterval(), stretch=AsinhStretch(a=0.1))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# ax.imshow(the_differenceimage.image.array, origin="lower", cmap="gray")
# ax.imshow(asinh_img, origin="lower", cmap="gray")
ax.imshow(img, origin="lower", cmap="gray", norm=norm)

for src in sources:
    fp = src.getFootprint()
    if fp is None:
        continue

    box = fp.getBBox()
    ax.add_patch(
        plt.Rectangle(
            (box.getMinX(), box.getMinY()),
            box.getWidth(),
            box.getHeight(),
            fill=False,
            edgecolor="red",
            linewidth=1,
        )
    )

plt.show()

In [ ]:
img = the_differenceimage.image.array

for src in sources[:10]:
    fp = src.getFootprint()
    if fp is None:
        continue

    bbox = fp.getBBox()
    sub = img[bbox.getMinY() : bbox.getMaxY() + 1, bbox.getMinX() : bbox.getMaxX() + 1]

    plt.figure()
    plt.imshow(sub, origin="lower")
    plt.title(f"Footprint for Source {src.getId()}")
    plt.colorbar()
    plt.show()

## The PSF Kernel


### 🎯 1. Estimateur PSF flux (forme scalaire)

$$F = \frac{\sum_i \frac{I_i P_i}{\sigma_i^2}}{\sum_i \frac{P_i^2}{\sigma_i^2}}$$

### 🎯 2. Formulation comme problème de moindres carrés

$$ F = \arg\min_F \sum_i \frac{(I_i - F P_i)^2}{\sigma_i^2}$$



###  🎯 3. Équation normale (condition du minimum)

$$\frac{\partial}{\partial F} \sum_i \frac{(I_i - F P_i)^2}{\sigma_i^2} = 0$$

ce qui donne :

$$\sum_i \frac{I_i P_i}{\sigma_i^2} = F \sum_i \frac{P_i^2}{\sigma_i^2}$$




### 🎯 4. Forme matricielle (LSST / AFW / meas_base)

$$F = \frac{\mathbf{P}^T \mathbf{W} \mathbf{I}}{\mathbf{P}^T \mathbf{W} \mathbf{P}}$$
avec :

$$\mathbf{W} = \mathbf{\Sigma}^{-1}$$
et souvent diagonale :
$$W_{ii} = \frac{1}{\sigma_i^2}$$

In [ ]:
psf = the_differenceimage.getPsf()

In [ ]:
type(psf)

In [ ]:
for src in sources[:10]:
    x = src.getX()
    y = src.getY()
    pos = geom.Point2D(x, y)
    kernel = psf.computeImage(pos)
    plt.figure()
    plt.imshow(kernel.array, origin="lower")
    plt.title(f"PSF Kernel for Source {src.getId()}")
    plt.colorbar()
    plt.show()

In [ ]:
# display.clearViewer()